In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 🤖 Model Training - Churn Prediction\n",
    "\n",
    "In this notebook, we will train and evaluate multiple Machine Learning models to predict customer churn."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Import libraries\n",
    "import sys\n",
    "sys.path.append('..')\n",
    "\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "import pickle\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "# Configuration\n",
    "plt.style.use('seaborn-v0_8')\n",
    "sns.set_palette(\"husl\")\n",
    "%matplotlib inline\n",
    "\n",
    "print(\"✅ Libraries imported successfully\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Import models and metrics\n",
    "from sklearn.linear_model import LogisticRegression\n",
    "from sklearn.ensemble import RandomForestClassifier\n",
    "from xgboost import XGBClassifier\n",
    "from sklearn.model_selection import cross_val_score, GridSearchCV\n",
    "from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, precision_recall_curve\n",
    "\n",
    "from src.data_processing import get_processed_data\n",
    "\n",
    "print(\"✅ Models and metrics imported\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 📥 Load and Process Data"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load and process data\n",
    "print(\"📥 Loading and processing data...\")\n",
    "data_dict = get_processed_data('../data/telco_churn.csv')\n",
    "\n",
    "X_train = data_dict['X_train']\n",
    "X_test = data_dict['X_test']\n",
    "y_train = data_dict['y_train']\n",
    "y_test = data_dict['y_test']\n",
    "feature_names = data_dict['feature_names']\n",
    "processor = data_dict['processor']\n",
    "\n",
    "print(f\"📊 Training dimensions: {X_train.shape}\")\n",
    "print(f\"📈 Test dimensions: {X_test.shape}\")\n",
    "print(f\"🎯 Features: {len(feature_names)}\")\n",
    "print(f\"📋 Class balance - Train: {y_train.mean():.3f}, Test: {y_test.mean():.3f}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 🚀 Model Training"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Define models\n",
    "models = {\n",
    "    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),\n",
    "    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),\n",
    "    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss', n_estimators=100)\n",
    "}\n",
    "\n",
    "# Train and evaluate models\n",
    "results = {}\n",
    "\n",
    "print(\"\\n\" + \"=\"*50)\n",
    "print(\"MODEL TRAINING\")\n",
    "print(\"=\"*50)\n",
    "\n",
    "for name, model in models.items():\n",
    "    print(f\"\\n🚀 Training {name}...\")\n",
    "    \n",
    "    # Train model\n",
    "    model.fit(X_train, y_train)\n",
    "    \n",
    "    # Predict\n",
    "    y_pred = model.predict(X_test)\n",
    "    y_pred_proba = model.predict_proba(X_test)[:, 1]\n",
    "    \n",
    "    # Calculate metrics\n",
    "    accuracy = model.score(X_test, y_test)\n",
    "    auc_score = roc_auc_score(y_test, y_pred_proba)\n",
    "    \n",
    "    # Cross-validation\n",
    "    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc')\n",
    "    \n",
    "    results[name] = {\n",
    "        'model': model,\n",
    "        'accuracy': accuracy,\n",
    "        'auc_score': auc_score,\n",
    "        'cv_mean': cv_scores.mean(),\n",
    "        'cv_std': cv_scores.std(),\n",
    "        'y_pred': y_pred,\n",
    "        'y_pred_proba': y_pred_proba\n",
    "    }\n",
    "    \n",
    "    print(f\"  ✅ Accuracy: {accuracy:.4f}\")\n",
    "    print(f\"  📊 AUC Score: {auc_score:.4f}\")\n",
    "    print(f\"  🔄 CV AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 📊 Model Comparison"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Compare models\n",
    "print(\"\\n\" + \"=\"*50)\n",
    "print(\"MODEL COMPARISON\")\n",
    "print(\"=\"*50)\n",
    "\n",
    "results_df = pd.DataFrame({\n",
    "    'Model': list(results.keys()),\n",
    "    'Accuracy': [results[name]['accuracy'] for name in results.keys()],\n",
    "    'AUC Score': [results[name]['auc_score'] for name in results.keys()],\n",
    "    'CV AUC Mean': [results[name]['cv_mean'] for name in results.keys()],\n",
    "    'CV AUC Std': [results[name]['cv_std'] for name in results.keys()]\n",
    "}).sort_values('AUC Score', ascending=False)\n",
    "\n",
    "display(results_df.style.background_gradient(cmap='Blues'))\n",
    "\n",
    "# Select the best model\n",
    "best_model_name = results_df.iloc[0]['Model']\n",
    "best_model = results[best_model_name]['model']\n",
    "best_result = results[best_model_name]\n",
    "\n",
    "print(f\"\\n🌟 BEST MODEL: {best_model_name}\")\n",
    "print(f\"🎯 AUC Score: {best_result['auc_score']:.4f}\")\n",
    "print(f\"📈 Accuracy: {best_result['accuracy']:.4f}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 📈 Visualization of Results"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Visualization of results\n",
    "plt.figure(figsize=(15, 5))\n",
    "\n",
    "# Confusion matrix\n",
    "plt.subplot(1, 3, 1)\n",
    "cm = confusion_matrix(y_test, best_result['y_pred'])\n",
    "sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', \n",
    "            xticklabels=['No Churn', 'Churn'], \n",
    "            yticklabels=['No Churn', 'Churn'])\n",
    "plt.title(f'Confusion Matrix - {best_model_name}')\n",
    "plt.xlabel('Predicted')\n",
    "plt.ylabel('True')\n",
    "\n",
    "# ROC Curve\n",
    "plt.subplot(1, 3, 2)\n",
    "fpr, tpr, _ = roc_curve(y_test, best_result['y_pred_proba'])\n",
    "plt.plot(fpr, tpr, linewidth=2, label=f'{best_model_name} (AUC = {best_result[\"auc_score\"]:.3f})')\n",
    "plt.plot([0, 1], [0, 1], 'k--')\n",
    "plt.xlabel('False Positive Rate')\n",
    "plt.ylabel('True Positive Rate')\n",
    "plt.title('ROC Curve')\n",
    "plt.legend()\n",
    "plt.grid(True, alpha=0.3)\n",
    "\n",
    "# Feature Importance (if tree-based)\n",
    "plt.subplot(1, 3, 3)\n",
    "if hasattr(best_model, 'feature_importances_'):\n",
    "    feature_importance = pd.DataFrame({\n",
    "        'feature': feature_names,\n",
    "        'importance': best_model.feature_importances_\n",
    "    }).sort_values('importance', ascending=True).tail(10)\n",
    "    \n",
    "    plt.barh(feature_importance['feature'], feature_importance['importance'])\n",
    "    plt.title('Top 10 Most Important Features')\n",
    "    plt.xlabel('Importance')\n",
    "else:\n",
    "    # For Logistic Regression, use absolute coefficients\n",
    "    feature_importance = pd.DataFrame({\n",
    "        'feature': feature_names,\n",
    "        'importance': np.abs(best_model.coef_[0])\n",
    "    }).sort_values('importance', ascending=True).tail(10)\n",
    "    \n",
    "    plt.barh(feature_importance['feature'], feature_importance['importance'])\n",
    "    plt.title('Top 10 Coefficients (absolute)')\n",
    "    plt.xlabel('Coefficient Magnitude')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 📋 Classification Report"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Detailed classification report\n",
    "print(\"\\n\" + \"=\"*50)\n",
    "print(\"DETAILED CLASSIFICATION REPORT\")\n",
    "print(\"=\"*50)\n",
    "print(classification_report(y_test, best_result['y_pred'], \n",
    "                            target_names=['No Churn', 'Churn']))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 🔍 SHAP Analysis for Explainability"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# SHAP analysis for explainability\n",
    "print(\"\\n\" + \"=\"*50)\n",
    "print(\"SHAP ANALYSIS - MODEL EXPLAINABILITY\")\n",
    "print(\"=\"*50)\n",
    "\n",
    "try:\n",
    "    import shap\n",
    "    \n",
    "    # Train explainer\n",
    "    explainer = shap.TreeExplainer(best_model)\n",
    "    \n",
    "    # Calculate SHAP values for a data subset\n",
    "    sample_idx = np.random.choice(len(X_test), size=min(500, len(X_test)), replace=False)\n",
    "    X_sample = X_test.iloc[sample_idx]\n",
    "    \n",
    "    shap_values = explainer.shap_values(X_sample)\n",
    "    \n",
    "    # Global importance plot\n",
    "    plt.figure(figsize=(10, 8))\n",
    "    shap.summary_plot(shap_values, X_sample, feature_names=feature_names, show=False)\n",
    "    plt.title('Feature Importance - SHAP')\n",
    "    plt.tight_layout()\n",
    "    plt.show()\n",
    "    \n",
    "    # Bar plot with mean importance\n",
    "    plt.figure(figsize=(10, 6))\n",
    "    shap.summary_plot(shap_values, X_sample, feature_names=feature_names, plot_type=\"bar\", show=False)\n",
    "    plt.title('Mean Feature Importance - SHAP')\n",
    "    plt.tight_layout()\n",
    "    plt.show()\n",
    "    \n",
    "    print(\"✅ SHAP analysis completed successfully\")\n",
    "    \n",
    "    # Save explainer for the Streamlit app\n",
    "    with open('../models/shap_explainer.pkl', 'wb') as f:\n",
    "        pickle.dump(explainer, f)\n",
    "    print(\"💾 SHAP explainers saved successfully\")\n",
    "    \n",
    "except ImportError:\n",
    "    print(\"⚠️ SHAP not installed. Install with: pip install shap\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 💰 Business Impact Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# DETAILED BUSINESS IMPACT ANALYSIS\n",
    "print(\"\\n\" + \"=\"*60)\n",
    "print(\"DETAILED BUSINESS IMPACT ANALYSIS\")\n",
    "print(\"=\"*60)\n",
    "\n",
    "# Calculate business metrics\n",
    "precision, recall, thresholds = precision_recall_curve(y_test, best_result['y_pred_proba'])\n",
    "\n",
    "# Find the best threshold based on F1-score\n",
    "f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)\n",
    "best_threshold = thresholds[np.argmax(f1_scores)]\n",
    "\n",
    "print(f\"🎯 Best threshold to maximize F1-score: {best_threshold:.3f}\")\n",
    "\n",
    "# Apply the best threshold\n",
    "optimized_predictions = (best_result['y_pred_proba'] > best_threshold).astype(int)\n",
    "\n",
    "# Business metrics with optimized threshold\n",
    "tn, fp, fn, tp = confusion_matrix(y_test, optimized_predictions).ravel()\n",
    "\n",
    "print(f\"\\n📈 METRICS WITH OPTIMIZED THRESHOLD:\")\n",
    "print(f\"  • True Positives: {tp} (correctly identified at-risk customers)\")\n",
    "print(f\"  • False Positives: {fp} (unnecessarily retained customers)\")\n",
    "print(f\"  • False Negatives: {fn} (lost customers not identified)\")\n",
    "\n",
    "# Estimate ROI\n",
    "avg_monthly_revenue = 70  # Average monthly revenue per customer\n",
    "retention_cost = 25       # Cost of retention campaign per customer\n",
    "customer_lifetime = 12    # Average customer lifetime in months\n",
    "\n",
    "potential_revenue_saved = tp * avg_monthly_revenue * customer_lifetime\n",
    "retention_campaign_cost = (tp + fp) * retention_cost\n",
    "net_roi = potential_revenue_saved - retention_campaign_cost\n",
    "\n",
    "print(f\"\\n💰 ESTIMATED FINANCIAL ANALYSIS:\")\n",
    "print(f\"  • Potentially saved revenue: ${potential_revenue_saved:,.2f}\")\n",
    "print(f\"  • Retention campaign cost: ${retention_campaign_cost:,.2f}\")\n",
    "print(f\"  • Estimated Net ROI: ${net_roi:,.2f}\")\n",
    "print(f\"  • Percentage ROI: {(net_roi/retention_campaign_cost)*100:.1f}%\")\n",
    "\n",
    "# Segment customers by risk\n",
    "risk_segments = pd.cut(best_result['y_pred_proba'], \n",
    "                       bins=[0, 0.3, 0.7, 1.0], \n",
    "                       labels=['Low Risk', 'Medium Risk', 'High Risk'])\n",
    "\n",
    "segment_counts = risk_segments.value_counts()\n",
    "print(f\"\\n🎯 CUSTOMER RISK SEGMENTATION:\")\n",
    "for segment, count in segment_counts.items():\n",
    "    percentage = count / len(risk_segments) * 100\n",
    "    print(f\"  • {segment}: {count} customers ({percentage:.1f}%)\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 💾 Save Model and Results"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Save the best model and processor\n",
    "print(f\"\\n💾 Saving the best model ({best_model_name})...\")\n",
    "\n",
    "# Save model and processor\n",
    "model_data = {\n",
    "    'model': best_model,\n",
    "    'processor': processor,\n",
    "    'feature_names': feature_names,\n",
    "    'results': results,\n",
    "    'best_threshold': best_threshold,\n",
    "    'X_test': X_test,\n",
    "    'y_test': y_test\n",
    "}\n",
    "\n",
    "with open('../models/best_model.pkl', 'wb') as f:\n",
    "    pickle.dump(model_data, f)\n",
    "\n",
    "# Save processor separately\n",
    "processor.save_processor('../models/data_processor.pkl')\n",
    "\n",
    "print(\"✅ Model and processor saved successfully in '../models/'\")\n",
    "\n",
    "# Final summary\n",
    "print(\"\\n\" + \"=\"*60)\n",
    "print(\"🎉 TRAINING COMPLETED SUCCESSFULLY\")\n",
    "print(\"=\"*60)\n",
    "print(f\"📊 Final Model: {best_model_name}\")\n",
    "print(f\"🎯 AUC Score: {best_result['auc_score']:.4f}\")\n",
    "print(f\"📈 Accuracy: {best_result['accuracy']:.4f}\")\n",
    "print(f\"💰 Estimated ROI: ${net_roi:,.2f}\")\n",
    "print(f\"🔧 High-Risk Customers Identified: {tp}\")\n",
    "print(\"\\n🚀 The model is ready to be used in the Streamlit application!\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.8.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}